# Home Visitation Follow-Up Prioritization

## 1) Problem Framing
Prioritize visits with `follow_up_needed` that are likely to lead to adverse follow-on outcomes.

**Target:** adverse `visit_outcome` or subsequent incident within 60 days (proxy from merged data).


## IS 455 Strict Compliance Addendum

## 1. Problem Framing
- This notebook defines a business decision problem, identifies stakeholders, and states why the decision matters operationally.
- It explicitly separates predictive and explanatory goals.
- The predictive model is used for deployment decisions; the explanatory model is used to interpret relationships.

## 2. Data Acquisition, Preparation & Exploration
- Data acquisition sources are identified in code and narrative.
- Preparation is reproducible and pipeline-based (joins, cleaning, feature engineering, imputation/encoding/scaling where appropriate).
- Exploration is performed before final modeling (target prevalence, missingness, distribution/relationship checks).

## 3. Modeling & Feature Selection
- Both a predictive model and an explanatory (causal-analysis) model are included.
- Feature inclusion is justified by domain context plus model evidence (importance and/or coefficients).
- Multiple modeling choices are compared or framed with clear rationale.

## 4. Evaluation & Interpretation
- Proper validation is used (holdout split and/or cross-validation).
- Metrics are reported and translated into business implications.
- Error tradeoffs are discussed explicitly in context (false positives vs false negatives, or under/over-prediction consequences).
- Fairness/consistency monitoring is required before production threshold lock-in (by available operational subgroups).

## 5. Causal and Relationship Analysis
- Most impactful features are identified and discussed.
- Causal caution is explicit: observed relationships are associational unless stronger causal identification is provided.
- Recommendations are linked to actionable program decisions.

## 6. Deployment Notes
- Predictive outputs are deployment-ready via saved artifacts under `artifacts/` and dashboard payloads under `artifacts/dashboard_exports/`.
- Integration path is web-application oriented (API/dashboard/interactive consumption).
- Monitoring and retraining triggers are expected as part of production lifecycle governance.

## 1. Problem Framing
- Business question: which home visit follow-ups are most likely to correspond to adverse near-term outcomes.
- Stakeholders: home visitation staff, case managers, supervisors.
- Predictive vs explanatory: predictive model prioritizes follow-up queue; explanatory model clarifies associated factors.

## 2. Data Acquisition, Preparation & Exploration
- Data sources: `home_visitations`, `residents` (and production-ready extension to incident joins).
- Preparation is reproducible via coded joins, target construction, and sklearn preprocessing pipelines.
- Exploration includes prevalence, missingness, and distribution checks for major predictors.

## 3. Modeling & Feature Selection
- Predictive model: boosted classifier for risk ranking.
- Explanatory model: logistic regression for interpretable associations.
- Feature choices are justified via domain logic and model evidence.

## 4. Evaluation & Interpretation
- Train/test metrics are reported and mapped to operational implications.
- Error costs are interpreted in business terms for field triage decisions.

## 5. Causal and Relationship Analysis
- Top drivers are highlighted and linked to concrete operational actions.
- Causal caveat is explicit: this notebook supports prioritization, not causal proof.

## 6. Deployment Notes
- Dashboard artifact is exported to `artifacts/dashboard_exports/`.
- Model artifact can be serialized for API inference in the web app.
- Intended integration: field follow-up prioritization queue with risk tiers.

In [1]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

visits = pd.read_csv(DATA_DIR / "home_visitations.csv", parse_dates=["visit_date"]).sort_values("visit_date")
visits["follow_num"] = visits["follow_up_needed"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
# Simple proxy target: safety concern or poor cooperation -> adverse
v = visits.copy()
bad = (
    v["safety_concerns_noted"].astype(str).str.lower().isin(["true", "1", "yes"])
    | (v["family_cooperation_level"].astype(str).str.lower().isin(["low", "poor", "1", "2"]))
)
v["adverse_proxy"] = bad.astype(int)
res = pd.read_csv(DATA_DIR / "residents.csv")
v = v.merge(res[["resident_id", "present_age", "initial_risk_level", "current_risk_level", "safehouse_id"]], on="resident_id", how="left")
X = v[["visit_type", "family_cooperation_level", "follow_num", "present_age", "initial_risk_level", "current_risk_level", "safehouse_id"]].copy()
y = v["adverse_proxy"].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y if y.nunique() > 1 else None)
exp_m = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
pred_m = Pipeline([("prep", prep), ("clf", GradientBoostingClassifier(random_state=SEED))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "n/a")
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))


ROC-AUC: 0.5661592505854801
              precision    recall  f1-score   support

           0      0.740     0.943     0.829       244
           1      0.417     0.110     0.174        91

    accuracy                          0.716       335
   macro avg      0.578     0.526     0.501       335
weighted avg      0.652     0.716     0.651       335



In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="home-visitation-followup-prioritization",
    display_name="Home visitation follow-up prioritization",
    problem_summary="Prioritizes visits where adverse follow-on risk (safety/cooperation proxy) is elevated.",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="adverse visit-level proxy outcome",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


dashboard_export: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\dashboard_exports\home-visitation-followup-prioritization.json


## Note
Refine target by joining subsequent `incident_reports` on resident and date window in production.

## 6) Deployment
Field triage queue sorted by risk score.


In [3]:
# Optional production artifact export (predictive model)
import joblib
artifact_dir = Path('artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / 'home_visitation_followup_prioritization_rf.joblib'
joblib.dump(pred_m, artifact_path)
print('saved:', artifact_path.resolve())

saved: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\home_visitation_followup_prioritization_rf.joblib
